# spectraPyle GUI 

This notebook generates a validated configuration file for spectraPyle and launch the code. 

## Steps
1. Fill parameters below or LOAD existing GUI
2. Click **Validate Config**
3. Save configuration file to GUI/JSON/YAML (optional)
3. Click **Run spectraPyle** to start stacking

#### Mandatory actions -> (*), otherwise: default values

#### Developer: Salvatore Quai (salvatore.quai@unibo.it)
#### Version: 0.0.1


If validation fails → fix highlighted fields.


In [1]:
import json
import os
from pathlib import Path
import ipywidgets as w
from IPython.display import display, HTML,clear_output
from pydantic import TypeAdapter,BaseModel

import contextlib
import time
import threading

import importlib
import spectraPyle

import spectraPyle as stsp

import spectraPyle.stacking.stacking as stack
import spectraPyle.runtime.runtime_adapter  
import spectraPyle.schema.schema
importlib.reload(spectraPyle.runtime.runtime_adapter)  
importlib.reload(spectraPyle.schema.schema)  
importlib.reload(stack)
from spectraPyle.runtime.runtime_adapter import build_config_from_widgets, export_config_to_json, export_config_to_yaml
from spectraPyle.schema.schema import StackingConfigResolver
from spectraPyle.schema.schema import StackingConfig

In [2]:
PACKAGE_ROOT = Path(stsp.__file__).parent.parent.parent
#path_to_config_dir = PACKAGE_ROOT / "configs" / "GUI"
#print(path_to_config_dir)
#print (PACKAGE_ROOT)

In [3]:
def section(title):
    return w.HTML(f"<h3 style='color:#2c3e50'>{title}</h3>")

style = {'description_width': 'initial'}
#style = {'description_width': '300px'}

In [4]:
module_dir = os.path.dirname(spectraPyle.__file__)
rules_path = os.path.join(module_dir, "instruments", "instruments_rules.json")

with open(rules_path) as f:
    RULES = json.load(f)

In [5]:
# instrument:

instrument_w = w.Dropdown(
    options=["euclid", "desi"],
    value="euclid",
    description="Instrument", 
    style=style,
)

survey_w = w.Dropdown(description="Survey", style=style)
grism_w  = w.Dropdown(description="Grism", style=style)
datarelease_w  = w.Dropdown(description="Data release", style=style)

def update_instrument(*_):

    inst = instrument_w.value
    rules = RULES[inst]

    survey_w.options = list(rules["surveys"].keys())
    survey_w.value = rules["defaults"]["survey"]

    update_survey()

def update_survey(*_):

    inst = instrument_w.value
    survey = survey_w.value

    survey_rules = RULES[inst]["surveys"][survey]

    grism_w.options = survey_rules["grisms"]
    datarelease_w.options = survey_rules["data_release"]

    grism_w.value = RULES[inst]["defaults"]["grism"]
    datarelease_w.value = RULES[inst]["defaults"]["data_release"]

instrument_w.observe(update_instrument, names="value")
survey_w.observe(update_survey, names="value")

update_instrument()

In [6]:
# IO:

MODE_INDIVIDUAL = "individual fits"
MODE_METADATA = "metadata path"
MODE_COMBINED = "combined fits"

## format
spectra_format_w = w.RadioButtons(
    options=[MODE_INDIVIDUAL, MODE_COMBINED, MODE_METADATA],
    value=MODE_INDIVIDUAL,
    layout={'width': 'max-content'},
    description="Spectra format", style=style
)

## dir in:
dirIn_w = w.Text(
    placeholder="./spectraPyle/test_data/INDIVIDUAL_FITS/",
    description="Catalogue directory:", style=style
)

fileIn_w = w.Text(
    description="Catalogue filename:", style=style
)

fileExt_w = w.Dropdown(
    options=["fits", "csv", "npz"],
    value="csv",
    description="Catalogue extension", style=style
)

## dir out:
dirOut_w = w.Text(
    placeholder="./spectraPyle/test_data/INDIVIDUAL_FITS/output/",
    description="Output directory:", style=style
)


In [7]:
## OUTPUT FILENAME:
output_name_mode_w = w.RadioButtons(
    options=["AUTO", "custom"],
    value="AUTO",
    layout={'width': 'max-content'},
    description="Output filename", style=style
)

output_name_custom_w = w.Text(
    placeholder="e.g. stacked_Halpha_Euclid_Q1",
    description="Custom name", style=style
)

output_name_w = w.Text(
    value="AUTO",  # special keyword → tells config to use filename_builder
    layout=w.Layout(display="none")
)

output_name_box = w.VBox()

def update_output_name(change=None):

    if output_name_mode_w.value == "AUTO":

        output_name_box.children = []

        # Pass keyword to config
        output_name_w.value = "AUTO"

    else:

        output_name_box.children = [output_name_custom_w]

        output_name_w.value = output_name_custom_w.value.strip()

output_name_mode_w.observe(update_output_name, names="value")
update_output_name()

In [8]:
spectra_dir_w = w.Text(
    description="Spectra Directory",
    placeholder="/data/spectra",
    #layout=w.Layout(display="none")
)

spectra_dir_box = w.VBox([spectra_dir_w])


def update_spectra_dir(change=None):

    mode = spectra_format_w.value

    if mode == MODE_INDIVIDUAL or mode == MODE_COMBINED:
        spectra_dir_box.layout.display = ""
        spectra_dir_w.disabled = False
    
        

    else:
        spectra_dir_box.layout.display = "none"



In [9]:
spectra_datafile_w = w.Text(
    description="Spectra filename (FITS)",
    style=style,
    placeholder="COMBINED_spectra",
)

spectra_datafile_box = w.VBox(
    [spectra_datafile_w],
    layout=w.Layout(display="none")   # hidden by default
)


def update_spectra_datafile(change=None):

    mode = spectra_format_w.value

    if mode == MODE_COMBINED:
        # visible + editable
        spectra_datafile_box.layout.display = ""
        spectra_datafile_w.disabled = False

    elif mode == MODE_METADATA:
        # hidden + forced value
        spectra_datafile_box.layout.display = "none"
        spectra_datafile_w.value = "metadata"

    else:
        # hidden + empty
        spectra_datafile_box.layout.display = "none"
        spectra_datafile_w.value = ""


In [10]:
# Physics:
## Cosmology:
cosmo_H0 = w.FloatText(
    value=70,
    description=r"H$_0$"
)

cosmo_Om0 = w.FloatText(
    value=0.3,
    description="$\Omega_0$"
)




In [11]:
# Physics
## Redhift

ztype_w = w.Dropdown(
    options=[
        ("Rest frame", "rest_frame"),
        ("Observed frame", "observed_frame"),
        ("Minimum z", "minimum_z"),
        ("Maximum z", "maximum_z"),
        ("Median z", "median_z"),
        ("Custom value", "custom")
    ],
    value="rest_frame",
    description="Redshift type",
    style=style
)

z_custom_w = w.FloatText(
    value=None,
    description="Custom z",
    style=style
)

z_custom_box = w.VBox()

def update_ztype(change=None):

    if ztype_w.value == "custom":
        z_custom_box.children = [z_custom_w]
    else:
        z_custom_box.children = []
        
ztype_w.observe(update_ztype, names="value")
#update_ztype()


In [12]:
# Physics:
## Normalization
norm_w = w.Dropdown(
    options=["no_normalization", "custom", "median", "interval", "integral", "template"],
    value="median",
    description="Normalization",
    style=style
)

conservation_w = w.Dropdown(
    options=["flux", "luminosity"],
    value="luminosity",
    description="Conservation",
    style=style,
    layout=w.Layout(display="none")
)

def update_normalization(change=None):

    if norm_w.value == "no_normalization":
        # Must choose conservation
        conservation_w.layout.display = ""
        conservation_w.disabled = False

    else:
        # Must be None
        conservation_w.layout.display = "none"

#norm_w.observe(update_normalization, names="value")
#update_normalization()



In [13]:
## Interval wavelength for interval normalization, and statistic to apply to the flux in the interval:

lambda_norm_rest_value = None
interval_norm_statistics_value = None
conservation_value = None

lambda_min_w = w.FloatText(
    value=None,
    description="λ min",
    style=style
)

lambda_max_w = w.FloatText(
    value=None,
    description="λ max",
    style=style
)

interval_stat_w = w.Dropdown(
    options=["median", "mean", "maximum", "minimum"],
    value="median",
    description="Statistic",
    style=style
)

regular_box = w.VBox([
    w.HTML("<b>Flux conservation mode</b>"),
    conservation_w
])

interval_box = w.VBox([
    w.HTML("<b>Normalization wavelength interval (rest-frame)</b>"),
    w.HBox([lambda_min_w, lambda_max_w]),
    interval_stat_w
])

norm_dynamic_box = w.VBox()

def update_norm_ui(change=None):

    global lambda_norm_rest_value
    global interval_norm_statistics_value
    global conservation_value

    mode = norm_w.value

    # ---------- NO Normalization ----------
    if mode == "no_normalization":

        norm_dynamic_box.children = [regular_box]

        conservation_value = conservation_w.value
        lambda_norm_rest_value = None
        interval_norm_statistics_value = None

    # ---------- INTERVAL ----------
    elif mode == "interval":

        norm_dynamic_box.children = [interval_box]

        conservation_value = None
        lambda_norm_rest_value = [
            float(lambda_min_w.value),
            float(lambda_max_w.value)
        ]
        interval_norm_statistics_value = interval_stat_w.value

    # ---------- OTHER MODES ----------
    else:

        norm_dynamic_box.children = []

        conservation_value = None
        lambda_norm_rest_value = None
        interval_norm_statistics_value = None


def update_regular_values(change=None):
    global conservation_value

    if norm_w.value == "no_normalization":
        conservation_value = conservation_w.value

def update_interval_values(change=None):

    global lambda_norm_rest_value
    global interval_norm_statistics_value

    if norm_w.value == "interval":
        lambda_norm_rest_value = [
            float(lambda_min_w.value),
            float(lambda_max_w.value)
        ]
        interval_norm_statistics_value = interval_stat_w.value

        
## observers:
#norm_w.observe(update_norm_ui, names="value")

conservation_w.observe(update_regular_values, names="value")
update_regular_values()

lambda_min_w.observe(update_interval_values, names="value")
update_interval_values()
lambda_max_w.observe(update_interval_values, names="value")
update_interval_values()
interval_stat_w.observe(update_interval_values, names="value")
update_interval_values()


In [14]:
## Sampling

resampling_w = w.Dropdown(
    options=[
        ("Linear λ sampling", "lambda"),
        ("Log λ sampling", "log_lambda"),
        ("Shifted λ sampling", "lambda_shifted"),
        ("No resampling (observed frame only)", None)
    ],
    value="lambda",
    description="Resampling",
    style=style
)

resampling_note = w.HTML(
    "<i>Note: None allowed only in observed_frame and requires identical wavelength grids.</i>"
)


pixel_type_w = w.RadioButtons(
    options=[
        ("Manual pixel size", "manual"),
        ("Instrumental resolution", "instrumental")
    ],
    value="manual",
    description="Pixel mode",
    style=style
)

instrumental_resolution_note = w.HTML(
    "<i> Note: 'Instrumental resolution': the pixel size will be calculated according to the instrumental resolution provided in the instrument file.</i>"
)

pixel_size_w = w.FloatText(
    value=6,
    description="Δλ [Å]",
    style=style
)

n_nyq_w = w.FloatText(
    value=5,
    description="Nyquist sampling N",
    style=style
)

#pixelSize_type_w = w.Text(layout=w.Layout(display="none"))
#pixelSize_value_w = w.Text(layout=w.Layout(display="none"))
#n_nyq_value_w = w.Text(layout=w.Layout(display="none"))

pixel_size_mode_value = None       
pixel_size_value = None            
n_nyq_value = None                 

## containers:
pixel_manual_box = w.VBox([pixel_size_w])
pixel_instr_box = w.VBox([n_nyq_w])

pixel_dynamic_box = w.VBox()
resampling_dynamic_box = w.VBox()

def update_resampling(change=None):

    global pixel_size_mode_value
    global pixel_size_value
    global n_nyq_value

    # Enforce ztype rule
    if ztype_w.value != "observed_frame" and resampling_w.value is None:
        resampling_w.value = "lambda"
        return

    if resampling_w.value is None:

        resampling_dynamic_box.children = []

        pixel_size_mode_value = None
        pixel_size_value = None
        n_nyq_value = None

    else:

        resampling_dynamic_box.children = [pixel_type_w, pixel_dynamic_box]
        update_pixel_mode()


def update_pixel_mode(change=None):

    global pixel_size_mode_value
    global pixel_size_value
    global n_nyq_value

    if resampling_w.value is None:
        return

    if pixel_type_w.value == "manual":

        pixel_dynamic_box.children = [pixel_manual_box]

        pixel_size_mode_value = "manual"
        pixel_size_value = float(pixel_size_w.value)
        n_nyq_value = None

    else:

        pixel_dynamic_box.children = [pixel_instr_box]

        pixel_size_mode_value = "instrumental"
        pixel_size_value = None
        n_nyq_value = float(n_nyq_w.value)


def update_manual_value(change=None):

    global pixel_size_value

    if pixel_type_w.value == "manual":
        pixel_size_value = float(pixel_size_w.value)

def update_instr_value(change=None):

    global n_nyq_value

    if pixel_type_w.value == "instrumental":
        n_nyq_value = float(n_nyq_w.value)


        
## observers:
resampling_w.observe(update_resampling, names="value")
#ztype_w.observe(update_resampling, names="value")

pixel_type_w.observe(update_pixel_mode, names="value")

pixel_size_w.observe(update_manual_value, names="value")
n_nyq_w.observe(update_instr_value, names="value")

update_resampling()



In [15]:
## ------- REFINING WAVELENGTH EXTENT (optional) ------- ##
lambda_banner = w.HTML("""
<div style="border-left:6px solid #2c7fb8; background:#eef6fb; padding:10px;">
Optional wavelength restriction for the stacked spectrum.<br><br>

If enabled, the stacked spectrum will only include wavelengths between:<br>
<b>left_edge × (1+z_stacking)</b> and <b>right_edge × (1+z_stacking)</b><br><br>

Useful to focus on emission lines or specific spectral regions.
</div>
""")

lambda_enable_w = w.Checkbox(
    value=False,
    description="Limit wavelength range",
    style=style
)

left_edge_w = w.FloatText(
    value=6150,
    description="Left edge",
    style=style
)

right_edge_w = w.FloatText(
    value=6750,
    description="Right edge",
    style=style
)

lambda_box = w.VBox()

def update_lambda_box(change=None):

    if lambda_enable_w.value:
        lambda_box.children = [
            lambda_banner,
            left_edge_w,
            right_edge_w
        ]
    else:
        lambda_box.children = []

# observer:
lambda_enable_w.observe(update_lambda_box, names="value")
update_lambda_box()

## ------- EDGE CROP (optional) ------- ##

edges_banner = w.HTML("""
<div style="border-left:6px solid #f28e2b; background:#fff4e6; padding:10px;">
Optional cropping of spectrum edges.<br><br>

Removes the first N and last M pixels before stacking.<br><br>

Uses Python slicing rules.<br>
Example: first=10, last=-10 → keeps spectrum[10:-10]
</div>
""")

edges_enable_w = w.Checkbox(
    value=False,
    description="Crop spectrum edges",
    style=style
)

first_pixel_w = w.IntText(
    value=10,
    description="First pixel",
    style=style
)

last_pixel_w = w.IntText(
    value=-10,
    description="Last pixel",
    style=style
)

edges_box = w.VBox()



def update_edges_box(change=None):

    if edges_enable_w.value:
        edges_box.children = [
            edges_banner,
            first_pixel_w,
            last_pixel_w
        ]
    else:
        edges_box.children = []

# Observer:
edges_enable_w.observe(update_edges_box, names="value")
update_edges_box()

In [16]:
# =========================================================
# Catalogue Column Mapping (CLEAN VERSION)
# =========================================================

# -------------------------
# Visible Widgets
# -------------------------

ID_column_w = w.Text(
    placeholder="e.g. object_id",
    description="ID column name",
    style=style
)

# --- Redshift ---
redshift_column_input_w = w.Text(
    placeholder="e.g. z",
    description="Redshift column name",
    style=style
)

redshift_box = w.VBox()

# --- Metadata ---
metadata_path_w = w.Text(
    placeholder="e.g. datalabs_path",
    description="Metadata path",
    style=style
)

metadata_file_w = w.Text(
    placeholder="e.g. file_name",
    description="Metadata file",
    style=style
)

metadata_indx_w = w.Text(
    placeholder="e.g. hdu_index",
    description="Metadata HDU",
    style=style
)

metadata_box = w.VBox()

# --- Galactic Extinction ---
gal_ext_corr_w = w.Checkbox(
    value=False,
    description="Galactic extinction correction"
)

gal_ext_name_input_w = w.Text(
    placeholder="e.g. gal_ebv",
    description="E(B-V) column name",
    style=style
)

gal_ext_box = w.VBox()

# --- Custom Norm ---
custom_norm_input_w = w.Text(
    placeholder="e.g. Hband_flux",
    description="Custom norm column name",
    style=style
)

custom_norm_box = w.VBox()

# --- Citation Banner ---
gal_ext_banner = w.HTML(
    """
    <div style="
        border-left: 6px solid #2c7fb8;
        background-color: #eef6fb;
        padding: 12px;
        margin-top: 10px;
        border-radius: 6px;
    ">
    <b>Galactic extinction corrections</b><br><br>

    Uses <b>Gordon+23 extinction curve</b> via <i>dust_extinction</i>.<br><br>

    Please cite the G23 model if used:<br>
    <a href="https://dust-extinction.readthedocs.io/en/latest/dust_extinction/references.html"
       target="_blank">
       dust_extinction reference page
    </a>
    </div>
    """,
    layout=w.Layout(display="none")
)

# -------------------------
# Redshift UI Logic
# -------------------------
def update_redshift_column(change=None):

    if ztype_w.value != "observed_frame":
        redshift_box.children = [redshift_column_input_w]
    else:
        redshift_box.children = []


# -------------------------
# Metadata UI Logic
# -------------------------
def update_metadata(change=None):

    if spectra_format_w.value == MODE_METADATA:

        metadata_box.children = [
            w.HTML("<b>Metadata column names mapping</b>"),
            metadata_path_w,
            metadata_file_w,
            metadata_indx_w
        ]

    else:
        metadata_box.children = []


# -------------------------
# Galactic Extinction UI
# -------------------------
def update_gal_ext(change=None):

    if gal_ext_corr_w.value:

        gal_ext_box.children = [gal_ext_name_input_w]
        gal_ext_banner.layout.display = "block"

    else:

        gal_ext_box.children = []
        gal_ext_banner.layout.display = "none"


# -------------------------
# Custom Norm UI
# -------------------------
def update_custom_norm(change=None):

    if norm_w.value == "custom":

        custom_norm_box.children = [
            w.HTML("<i>Make sure custom column exists in catalogue</i>"),
            custom_norm_input_w
        ]

    else:
        custom_norm_box.children = []
        

# -------------------------
# Observers
# -------------------------

ztype_w.observe(update_redshift_column, names="value")
spectra_format_w.observe(update_metadata, names="value")
gal_ext_corr_w.observe(update_gal_ext, names="value")
norm_w.observe(update_custom_norm, names="value")


# -------------------------
# Initial Sync
# -------------------------

update_redshift_column()
update_metadata()
update_gal_ext()
update_custom_norm()

'''display(
    ID_column_w,
    redshift_box,
    metadata_box,
    gal_ext_corr_w,
    gal_ext_box,
    gal_ext_banner,
    custom_norm_box
)
'''

'display(\n    ID_column_w,\n    redshift_box,\n    metadata_box,\n    gal_ext_corr_w,\n    gal_ext_box,\n    gal_ext_banner,\n    custom_norm_box\n)\n'

In [17]:
INSTRUMENT_EXTRA_QC = {
    "euclid": {
        "bad_pixels": True,
        "dithers": True,
    },
    "desi": {
        "bad_pixels": False,
        "dithers": False,
    },
    # future instruments:
    # "roman": {"bad_pixels": True, "dithers": False}
}

sigma_w = w.FloatSlider(
    value=4.,
    min=0.,
    max=5.,
    step=0.25,
    description="Sigma Clip", style=style
)


# bad pixels
'''
badpix_banner = w.HTML("""
<div style="border-left:6px solid #2c7fb8; background:#eef6fb; padding:10px;">
Only if available in the data structure of the chosen file<br><br>

If stacking spectra from another &lt;instrument>&gt;, check if a dedicated structure exists in:
<br>./spectraPyle/src/instruments/&lt;instrumentName&gt;.py
</div>
""")
'''

pixel_mask_select_w = w.SelectMultiple(
    options=list(range(7)),     # bits 0–7
    value=(0, 6),               # default Euclid recommendation
    rows=7,
    description="Bad pixel bits", style=style
)

bad_pixel_note = w.HTML(
    "<i> Note: Multiple values can be selected with shift and/or ctrl (or command) pressed and mouse clicks or arrow keys.</i>"
)

## dithers
dither_banner = w.HTML("""
<div style="border-left:6px solid #2c7fb8; background:#eef6fb; padding:10px;">
Minimum number of dithers in the coadded 1D spectrum.<br>
Higher is better.<br><br> Note: Euclid Q1 max dithers = 4 (recommended ≥ 2).
</div>
""")

dither_w = w.IntSlider(
    value=2,
    min=1,
    max=50,
    step=1,
    description="Min dithers", style=style
)

instrument_qc_box = w.VBox()

def update_instrument_qc(change=None):

    inst = instrument_w.value
    caps = INSTRUMENT_EXTRA_QC.get(inst, {})

    children = []

    # ---- BAD PIXELS ----
    if caps.get("bad_pixels", False):
        #children += [badpix_banner, bad_pixel_note, pixel_mask_select_w]
        children += [bad_pixel_note, pixel_mask_select_w]

    # ---- DITHERS ----
    if caps.get("dithers", False):
        children += [dither_banner, dither_w]

    instrument_qc_box.children = children

## observers:
instrument_w.observe(update_instrument_qc, names="value")
update_instrument_qc()



In [18]:
bootstrap_R = w.BoundedIntText(
    value=300,
    min=0,
    max=1000,
    description="Bootstrap R"
)


In [19]:
parallel_enabled = w.Checkbox(
    value=True,
    description="Enable Multiprocessing"
)

parallel_cpu_frac = w.FloatSlider(
    value=0.9,
    min=0.1,
    max=0.95,
    step=0.05,
    description="CPU Fraction"
)


In [20]:
plot_enabled = w.Checkbox(
    value=True,
    description="Plot stacked spectrum"
)

In [21]:
def update_all_spectra_format_w(change=None):
    update_spectra_dir()
    update_spectra_datafile()
    update_metadata()

def update_all_ztype_w(change=None):
    update_ztype()
    update_resampling()
    update_redshift_column()

def update_all_norm_w(change=None):
    update_normalization()
    update_norm_ui()
    update_custom_norm()

spectra_format_w.observe(update_all_spectra_format_w, names="value")
ztype_w.observe(update_all_ztype_w, names="value")
norm_w.observe(update_all_norm_w, names="value")


In [22]:
## Load exixting config
load_GUI_name = w.Text(description="Load  Path")

def restore_widgets_from_gui_dict(_):
    
    validation_load_gui.clear_output()
    
    path_to_config_dir = PACKAGE_ROOT / "configs" / "GUI"
    #path_to_config_dir = Path(os.getcwd()).parent / "configs" / "GUI"
    
    with validation_load_gui:
        print(f"Loading GUI: {load_GUI_name.value}.gui from path: {path_to_config_dir}")
    
    filename = load_GUI_name.value.strip()
    if not filename:
        with validation_load_gui:
            print ("Filename is empty")
        
        #raise ValueError("Filename is empty")

    if not filename.endswith(".gui"):
        filename += ".gui"

    full_path = path_to_config_dir / filename
    
    try:
        with open(full_path) as f:
            payload = json.load(f)

        # ---- Detect format ----
        if "gui_state" in payload:
            cfg = payload["gui_state"]
        else:
            # Old files: assume payload IS gui state
            cfg = payload

    except Exception as e:
        with validation_load_gui:
            print(f"Failed loading config: {e}")
        print(f"Failed loading config: {e}")
        return

    # ---- Restore widgets ----

    OBS_FREEZE = True
    
    
    def safe_set(widget, value):
        try:
            widget.value = value
        except Exception:
            pass
    
    def safe_set_2(widget, value):
        try:
            if isinstance(widget, (list, tuple)) and isinstance(value, (list, tuple)):
                if len(widget) == len(value):
                    for w, v in zip(widget, value):
                        w.value = v
            else:
                widget.value = value
        except Exception:
            pass
    
    # --------------------------------------------------
    #  Instrument FIRST (important hierarchy)
    # --------------------------------------------------
    safe_set(instrument_w, cfg.get("instrument_name", instrument_w.value))
    update_instrument()

    safe_set(survey_w, cfg.get("survey_name", survey_w.value))
    safe_set(grism_w, cfg.get("grism_type", grism_w.value))
    safe_set(datarelease_w, cfg.get("data_release", datarelease_w.value))

    # --------------------------------------------------
    #  IO
    # --------------------------------------------------
    safe_set(spectra_dir_w, cfg.get("spectra_dir", ""))
    safe_set(dirIn_w, cfg.get("input_dir", ""))
    safe_set(dirOut_w, cfg.get("output_dir", ""))

    safe_set(fileIn_w, cfg.get("filename_in", ""))
    safe_set(fileExt_w, cfg.get("filename_in_extention", "csv"))

    safe_set(spectra_format_w, cfg.get("spectra_mode", ""))
    safe_set(spectra_datafile_w, cfg.get("spectra_datafile", ""))

    # Output filename logic
    filename_out = cfg.get("filename_out", "AUTO")

    if filename_out == "AUTO":
        safe_set(output_name_mode_w, "default")
    else:
        safe_set(output_name_mode_w, "custom")
        safe_set(output_name_custom_w, filename_out)

    update_output_name()

    # --------------------------------------------------
    #  Catalog mapping
    # --------------------------------------------------
    safe_set(ID_column_w, cfg.get("ID_column_name", ""))
    safe_set(redshift_column_input_w, cfg.get("redshift_column_name", ""))

    safe_set(gal_ext_corr_w, cfg.get("galactic_extinction", False))
    safe_set(gal_ext_name_input_w, cfg.get("gal_ext_column_name", ""))

    safe_set(custom_norm_input_w, cfg.get("custom_column_name", ""))

    safe_set(metadata_path_w, cfg.get("metadata_path_column_name", ""))
    safe_set(metadata_file_w, cfg.get("metadata_file_column_name", ""))
    safe_set(metadata_indx_w, cfg.get("metadata_indx_column_name", ""))

    # --------------------------------------------------
    #  Physics
    # --------------------------------------------------
    safe_set(cosmo_H0, cfg.get("cosmo_H0", cosmo_H0.value))
    safe_set(cosmo_Om0, cfg.get("cosmo_Om0", cosmo_Om0.value))

    safe_set(ztype_w, cfg.get("z_type", ztype_w.value))
    safe_set(z_custom_w, cfg.get("z_value", z_custom_w.value)) 
    safe_set(norm_w, cfg.get("spectra_normalization", norm_w.value))

    safe_set(conservation_w, cfg.get("conservation", conservation_w.value))

    safe_set(interval_norm_statistics_value,
             cfg.get("interval_norm_statistics", interval_norm_statistics_value))

    #safe_set(lambda_norm_rest_value,
    #         cfg.get("lambda_norm_rest", lambda_norm_rest_value))
    
    safe_set_2(
        (lambda_min_w, lambda_max_w),
    cfg.get("lambda_norm_rest", lambda_norm_rest_value)
    )
    

    # --------------------------------------------------
    #  Resampling
    # --------------------------------------------------
    safe_set(resampling_w, cfg.get("pixel_resampling_type", resampling_w.value))
    safe_set(pixel_size_w, cfg.get("pixel_resampling", pixel_size_w.value))
    safe_set(pixel_type_w, cfg.get("pixel_size_type", pixel_type_w.value))
    safe_set(n_nyq_w, cfg.get("nyquist_sampling", n_nyq_w.value))

    # --------------------------------------------------
    #  QC / Performance
    # --------------------------------------------------
    safe_set(sigma_w, cfg.get("sigma_clipping_conditions", sigma_w.value))
    safe_set(bootstrap_R, cfg.get("bootstrapping_R", bootstrap_R.value))

    safe_set(pixel_mask_select_w,
             tuple(cfg.get("pixel_mask", list(pixel_mask_select_w.value))))

    safe_set(dither_w, cfg.get("n_min_dithers", dither_w.value))

    safe_set(parallel_enabled, cfg.get("multiprocessing", parallel_enabled.value))
    safe_set(parallel_cpu_frac, cfg.get("max_cpu_fraction", parallel_cpu_frac.value))

    # --------------------------------------------------
    #  Wavelength 
    # --------------------------------------------------
    lambda_edges = cfg.get("lambda_edges_rest", False)

    if lambda_edges:
        lambda_enable_w.value = True
        left_edge_w.value = lambda_edges[0]
        right_edge_w.value = lambda_edges[1]
    else:
        lambda_enable_w.value = False

    spectrum_edges = cfg.get("spectrum_edges", False)

    if spectrum_edges:
        edges_enable_w.value = True
        first_pixel_w.value = spectrum_edges[0]
        last_pixel_w.value = spectrum_edges[1]
    else:
        edges_enable_w.value = False
    
    safe_set(parallel_enabled, cfg.get("multiprocessing", parallel_enabled.value))

    safe_set(plot_enabled, cfg.get("plot_results", plot_enabled.value))

    # --------------------------------------------------
    # PHASE 8 — Force UI refresh once
    # --------------------------------------------------
    update_all_spectra_format_w()
    update_all_ztype_w()
    update_all_norm_w()
    update_gal_ext()
    
    with validation_load_gui:
        print("✅ GUI loaded successfully")


validation_load_gui = w.Output()
load_GUI_btn = w.Button(description="Load GUI", button_style="primary")

load_GUI_btn.on_click(restore_widgets_from_gui_dict)

#display(load_GUI_name, load_GUI_btn)

In [23]:
## preview config
def preview_config(_):
    with preview_out:
        clear_output()
        cfg = build_user_config()
        print(cfg)

def build_user_config():
    """ BUILD CONFIG FROM WIDGETS """
    
    lambda_edges_rest_value,spectrum_edges_value = build_wavelength_config()     
    
    return dict(
        ## IO
        spectra_dir = spectra_dir_w.value,
        input_dir = dirIn_w.value,
        output_dir = dirOut_w.value,

        filename_in = fileIn_w.value,
        filename_in_extention = fileExt_w.value,
        
        filename_out = output_name_w.value,
        
        spectra_mode = spectra_format_w.value,
        spectra_datafile = spectra_datafile_w.value,
        
        ID_column_name = ID_column_w.value,
        redshift_column_name = redshift_column_input_w.value,
        
        galactic_extinction = gal_ext_corr_w.value, 
        gal_ext_column_name = gal_ext_name_input_w.value,
        
        custom_column_name = custom_norm_input_w.value,
        
        metadata_path_column_name = metadata_path_w.value,
        metadata_file_column_name = metadata_file_w.value,
        metadata_indx_column_name = metadata_indx_w.value,

        ## INSTRUMENT
        instrument_name = instrument_w.value,
        survey_name = survey_w.value,
        grism_type = grism_w.value,
        data_release = datarelease_w.value,
        
        ## PHYSICS
        cosmo_H0 = cosmo_H0.value,
        cosmo_Om0 = cosmo_Om0.value,
        
        ## Redshift
        z_type = ztype_w.value,
        z_value = z_custom_w.value,
        
        spectra_normalization = norm_w.value,
        conservation=conservation_value,
        
        spectrum_edges=spectrum_edges_value,
        lambda_edges_rest=lambda_edges_rest_value,
        
        interval_norm_statistics=interval_norm_statistics_value,
        lambda_norm_rest=lambda_norm_rest_value,
        
        pixel_resampling_type = resampling_w.value,
        pixel_size_type = pixel_size_mode_value,
        pixel_resampling = pixel_size_value,
        nyquist_sampling = n_nyq_value,
        
        ## QUALITY and PERFORMANCE
        sigma_clipping_conditions = sigma_w.value,
        bootstrapping_R = bootstrap_R.value,
        
        pixel_mask = list(pixel_mask_select_w.value),
        n_min_dithers = dither_w.value,
        
        multiprocessing = parallel_enabled.value,
        max_cpu_fraction = parallel_cpu_frac.value,
        
        plot_results = plot_enabled.value,
    )

def build_wavelength_config():

    if lambda_enable_w.value:
        lambda_edges_rest = [
            float(left_edge_w.value),
            float(right_edge_w.value)
        ]
    else:
        lambda_edges_rest = None

    if edges_enable_w.value:
        spectrum_edges = [
            int(first_pixel_w.value),
            int(last_pixel_w.value)
        ]
    else:
        spectrum_edges = None

    return lambda_edges_rest, spectrum_edges


preview_out = w.Output()

preview_btn = w.Button(description="Preview Config",style=dict(button_color='orange'))
preview_btn.on_click(preview_config)

#display(preview_btn, preview_out)



In [24]:
validated_cfg = None

validate_btn = w.Button(
    description="Validate Config (*)",
    button_style="primary",
)

validation_output = w.Output()

def on_validate(_):
    global validated_cfg  
    
    validation_output.clear_output()
    
    try:
        with validation_output:
            cfg = build_config_from_widgets(build_user_config)
            cfg = StackingConfigResolver.resolve(cfg)
            print (f"validated config: {cfg}")
        
        validated_cfg = cfg
        
        with validation_output:
            print("✅ Config valid! Good Job!")
            print(f"Config version: {cfg.config_version}")
            
    except Exception as e:
        validated_cfg = None  
        with validation_output:
            print("❌ Validation failed!")
            print(e)

validate_btn.on_click(on_validate)


In [25]:
def on_export_GUI(_):
    global validated_cfg
    
    export_GUI_output.clear_output()
    
    path_to_config_dir = PACKAGE_ROOT / "configs" / "GUI"
    #path_to_config_dir = Path(os.getcwd()).parent / "configs" / "GUI"
    path_to_config_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        if validated_cfg is None:
            raise ValueError("No validated config available")
        
        gui_cfg = build_user_config()
        
        filename = save_GUI_name_w.value.strip()
        if not filename:
            raise ValueError("Filename is empty")

        if not filename.endswith(".gui"):
            filename += ".gui"

        full_path = path_to_config_dir / filename

        export_payload = {
            "gui_state": gui_cfg,
            "validated_config": validated_cfg.model_dump(mode="json")
        }

        with open(full_path, "w") as f:
            json.dump(export_payload, f, indent=2)

        with export_GUI_output:
            print(f"✅ GUI exported correctly: {full_path}")

    except Exception as e:
        with export_GUI_output:
            print("❌ Export GUI failed! see Log...")
        print("Export failed:", e)


export_GUI_output = w.Output()

save_GUI_name_w = w.Text(
    description="GUI filename",
    style=style
)

save_GUI_btn = w.Button(
    description="Save config (GUI)",
    style=dict(button_color='antiquewhite')
)

save_GUI_btn.on_click(on_export_GUI)


In [26]:
## common output name

save_config_name_w = w.Text(
    description="Config filename",
    style=style
)

In [27]:
def on_export_JSON(_):
    global validated_cfg
    
    export_JSON_output.clear_output()
    
    path_to_config_dir = PACKAGE_ROOT / "configs" / "JSON"
    #path_to_config_dir = Path(os.getcwd()).parent / "configs" / "JSON"
    path_to_config_dir.mkdir(parents=True, exist_ok=True)

    try:
        if validated_cfg is None:
            raise ValueError("No validated config available")

        filename = save_config_name_w.value.strip()
        if not filename:
            raise ValueError("Filename is empty")

        if not filename.endswith(".json"):
            filename += ".json"

        full_path = path_to_config_dir / filename

        export_config_to_json(validated_cfg, full_path)

        with export_JSON_output:
            print(f"✅ Export to JSON done: {full_path}")

    except Exception as e:
        with export_JSON_output:
            print("❌ Export to JSON failed! see Log...")
        print("Export failed:", e)

export_JSON_output = w.Output()

save_JSON_btn = w.Button(
    description="Save config (JSON)",
    style=dict(button_color='lightgreen')
)

save_JSON_btn.on_click(on_export_JSON)


In [28]:
def on_export_YAML(_):
    global validated_cfg
    
    export_YAML_output.clear_output()
    
    path_to_config_dir = PACKAGE_ROOT / "configs" / "YAML"
    #path_to_config_dir = Path(os.getcwd()).parent / "configs" / "YAML"
    path_to_config_dir.mkdir(parents=True, exist_ok=True)

    try:
        if validated_cfg is None:
            raise ValueError("No validated config available")

        filename = save_config_name_w.value.strip()
        if not filename:
            raise ValueError("Filename is empty")

        if not filename.endswith(".yaml"):
            filename += ".yaml"

        full_path = path_to_config_dir / filename

        export_config_to_yaml(validated_cfg, full_path)

        with export_YAML_output:
            print(f"✅ Export to YAML done: {full_path}")

    except Exception as e:
        with export_YAML_output:
            print("❌ Export to YAML failed! see Log...")
        print("Export failed:", e)

export_YAML_output = w.Output()

save_YAML_btn = w.Button(
    description="Save config (YAML)",
    style=dict(button_color='lightgreen')
)

save_YAML_btn.on_click(on_export_YAML)


In [29]:
"""def run_spectraPyle(_):
    global validated_cfg
    global stack
    
    run_spec_output.clear_output()
    try:
        if validated_cfg is None:
            with run_spec_output:
                print ("No validated config available")
            raise ValueError("No validated config available")
        
        with run_spec_output:
            print("------   Running spectraPyle ------")
            #gui_cfg = build_user_config()
            #stack_run = stack.main(gui_cfg) 
            importlib.reload(stack) 
            print(type(validated_cfg))
            stack_run = stack.main(validated_cfg)
            
            
    except Exception as e:
        with run_spec_output:
            print("❌ failed to run spectraPyle", e)
    
run_spec_output = w.Output()


run_spectraPyle_btn = w.Button(
    description="RUN spectraPyle",
    button_style="danger",
    #style=dict(
    #font_weight='bold',
    #text_decoration='underline')
)

run_spectraPyle_btn.on_click(run_spectraPyle)
"""

'def run_spectraPyle(_):\n    global validated_cfg\n    global stack\n\n    run_spec_output.clear_output()\n    try:\n        if validated_cfg is None:\n            with run_spec_output:\n                print ("No validated config available")\n            raise ValueError("No validated config available")\n\n        with run_spec_output:\n            print("------   Running spectraPyle ------")\n            #gui_cfg = build_user_config()\n            #stack_run = stack.main(gui_cfg) \n            importlib.reload(stack) \n            print(type(validated_cfg))\n            stack_run = stack.main(validated_cfg)\n\n\n    except Exception as e:\n        with run_spec_output:\n            print("❌ failed to run spectraPyle", e)\n\nrun_spec_output = w.Output()\n\n\nrun_spectraPyle_btn = w.Button(\n    description="RUN spectraPyle",\n    button_style="danger",\n    #style=dict(\n    #font_weight=\'bold\',\n    #text_decoration=\'underline\')\n)\n\nrun_spectraPyle_btn.on_click(run_spectraPyle

In [30]:
# Output panel
run_spec_output = w.Output()

# Progress widgets
progress_bar = w.IntProgress(
    value=0,
    min=0,
    max=100,
    description='Running:',
    bar_style='',  # '', 'success', 'danger'
    style={'bar_color': '#2196F3'},
    orientation='horizontal'
)

progress_label = w.HTML(value="Starting...")
final_report = w.HTML(value="")

def get_unique_log_path(base_path: Path, suffix: str = ".log") -> Path:
    """Generate unique filename: spectraPyle_run.log → spectraPyle_run_1.log → spectraPyle_run_2.log"""
    
    # Create parent directories if missing
    base_path.parent.mkdir(parents=True, exist_ok=True)
    
    if not base_path.exists():
        return base_path
    
    stem = base_path.stem  # "spectraPyle_run"
    counter = 1
    while True:
        new_path = base_path.parent / f"{stem}_{counter}{suffix}"
        if not new_path.exists():
            return new_path
        counter += 1

def run_spectraPyle(_):
    global validated_cfg
    global stack

    run_spec_output.clear_output()
    final_report.value = ""
    progress_bar.value = 0
    progress_bar.bar_style = ''
    progress_label.value = "Initializing..."

    if validated_cfg is None:
        with run_spec_output:
            print("No validated config available")
        return

    # Display UI
    with run_spec_output:
        display(progress_bar)
        display(progress_label)
        display(final_report)

    start_time = time.time()
    stop_flag = {"done": False}

    # --------------------------------------------------
    # Fake animated progress (indeterminate)
    # --------------------------------------------------
    def animate_progress():
        while not stop_flag["done"]:
            progress_bar.value = (progress_bar.value + 5) % 100
            elapsed = int(time.time() - start_time)
            progress_label.value = f"Running... {elapsed}s elapsed"
            time.sleep(0.5)

    thread = threading.Thread(target=animate_progress)
    thread.start()

    # --------------------------------------------------
    # Run pipeline
    # --------------------------------------------------
    
    try:
        #path_to_log_file = get_unique_log_path(
        #PACKAGE_ROOT / "logs" / "spectraPyle_run.log"
        #    )
        
        path_to_log_file = get_unique_log_path(
            Path(f"{dirOut_w.value}/spectraPyle_run.log")
        )

        print(f"Logging to: {path_to_log_file}")  # spectraPyle_run.log, _1, _2, etc.

        with open(path_to_log_file, "w") as logfile:
            with contextlib.redirect_stdout(logfile), \
                 contextlib.redirect_stderr(logfile):

                importlib.reload(stack)
                stack_run = stack.main(validated_cfg)

        stop_flag["done"] = True
        thread.join()

        progress_bar.value = 100
        progress_bar.bar_style = 'success'
        elapsed = round(time.time() - start_time, 2)

        final_report.value = f"""
        <div style="
            border:1px solid #4CAF50;
            padding:10px;
            border-radius:6px;
            background-color:#E8F5E9;">
            <b>spectraPyle finished successfully</b><br>
            Total runtime: {elapsed} seconds<br>
            Log file: <code>{path_to_log_file}</code>
        </div>
        """

    except Exception as e:
        stop_flag["done"] = True
        thread.join()

        progress_bar.bar_style = 'danger'
        elapsed = round(time.time() - start_time, 2)

        final_report.value = f"""
        <div style="
            border:1px solid #F44336;
            padding:10px;
            border-radius:6px;
            background-color:#FFEBEE;">
            <b>spectraPyle failed</b><br>
            Runtime before failure: {elapsed} seconds<br>
            Error: {e}<br>
            Check log file: <code>{path_to_log_file}</code>
        </div>
        """

run_spectraPyle_btn = w.Button(
    description="RUN spectraPyle",
    button_style="danger",
    #style=dict(
    #font_weight='bold',
    #text_decoration='underline')
)
        
run_spectraPyle_btn.on_click(run_spectraPyle)

In [31]:
# display:

display(
    section("spectraPyle GUI"),
    section("Load existing GUI"),
    load_GUI_name, 
    load_GUI_btn,
    validation_load_gui,
    
    section("Instrument"),
    instrument_w, 
    survey_w, 
    grism_w, 
    datarelease_w,
    instrument_qc_box,

    section("Input/Output (*)"),
    spectra_format_w,
    dirIn_w,
    fileIn_w,
    fileExt_w,
    spectra_dir_box,
    spectra_datafile_box,
    dirOut_w,
    output_name_mode_w,
    output_name_box,

    section("Cosmology"),
    cosmo_H0,
    cosmo_Om0,
    
    section("Redshift"),
    ztype_w,
    z_custom_box,
    
    section("Normalization"),
    norm_w,
    norm_dynamic_box,
    
    section("Resampling"),
    resampling_w,
    resampling_note,
    resampling_dynamic_box,
    instrumental_resolution_note,
    
    section("Refining wavelength extent (optional)"),
    lambda_enable_w,
    lambda_box,
    edges_enable_w,
    edges_box,
    
    section("Catalogue's column mapping (*)"),
    ID_column_w,
    redshift_box,
    metadata_box,
    gal_ext_corr_w,
    gal_ext_box,
    gal_ext_banner,
    custom_norm_box,
    
    section("Sigma clipping"),
    sigma_w,

    section("Bootstrap"),
    bootstrap_R,

    section("Parallel"),
    parallel_enabled,
    parallel_cpu_frac,
    
    section("Plot"),
    plot_enabled,

    section("Actions"),
    section("GUI preview"),
    preview_btn, 
    preview_out,
    section("Validate (*)"),
    validate_btn,
    validation_output,
    section("Export GUI"),
    save_GUI_name_w,
    save_GUI_btn,
    export_GUI_output,
    section("Export validated config (JSON, YAMLE)"),
    save_config_name_w,
    save_JSON_btn,
    export_JSON_output,
    save_YAML_btn,
    export_YAML_output,
    section("Run spectraPyle"),
    run_spectraPyle_btn,
    run_spec_output,
    
)




HTML(value="<h3 style='color:#2c3e50'>spectraPyle GUI</h3>")

HTML(value="<h3 style='color:#2c3e50'>Load existing GUI</h3>")

Text(value='', description='Load  Path')

Button(button_style='primary', description='Load GUI', style=ButtonStyle())

Output()

...

Button(description='Save config (YAML)', style=ButtonStyle(button_color='lightgreen'))

Output()

HTML(value="<h3 style='color:#2c3e50'>Run spectraPyle</h3>")

Button(button_style='danger', description='RUN spectraPyle', style=ButtonStyle())

Output()